**⚖️ 05 — Tổng hợp: yếu tố nào quyết định GIÁ và HỆ SỐ NHÂN?**

Gộp kết quả của 4 notebook trước bằng **cùng một thước đo**, để so sánh công bằng.

| Notebook | Yếu tố |
|---|---|
| `01_location` | 📍 Khu vực |
| `02_time` | 🕐 Giờ, thứ, cuối tuần |
| `03_weather` | 🌦️ Thời tiết |
| `04_distance_baseprice` | 📏 Quãng đường & loại xe |

> ⚠️ Tách riêng Uber/Lyft · kiểm soát `quãng đường × loại xe` · surge chỉ dùng Lyft.

**0. Nạp dữ liệu**

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, ".")
import importlib, _common; importlib.reload(_common)
from _common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt

setup()
df, dfU, dfL = load()

BINS = np.linspace(0, df.distance.max(), 16)
df["_ngay"] = (df.date_local - df.date_local.min()).dt.days

**1. 📊 Thước đo chung: mỗi yếu tố giải thích bao nhiêu % phổ giá?**

Cách đo: cố định `quãng đường` + `loại xe`, rồi xem yếu tố cần xét làm giá lệch bao nhiêu,
so với phổ giá thô ban đầu.

In [ ]:
YEU_TO = [("📏 Loại xe",      "name"),
          ("📍 Khu vực đón",  "source"),
          ("📍 Điểm đến",     "destination"),
          ("🕐 Giờ",          "hour_local"),
          ("🕐 Thứ",          "weekday_local"),
          ("🌦️ Thời tiết",    "short_summary"),
          ("🌦️ Nhiệt độ",     "temperature"),
          ("🌦️ Cường độ mưa", "precipIntensity")]

def suc_manh(dd, cot):
    """% pho gia tho ma cot nay giai thich duoc (da kiem soat q.duong + loai xe)."""
    b = pd.cut(dd.distance, BINS)
    tho = dd.groupby(b, observed=True).price.std().mean()
    if cot == "name":                      # loai xe: do truc tiep sau khi co dinh q.duong
        xe = dd.groupby([b, "name"], observed=True).price.std().mean()
        return (1 - xe/tho) * 100
    key = binned(dd[cot]).rename("_k")
    lech = dd.groupby([b, "name", key], observed=True).price.mean()              .groupby(level=[0, 1]).std().mean()
    return lech / tho * 100

rows = []
for lab, c in YEU_TO:
    rows.append({"Yếu tố": lab, "cột": c,
                 "% phổ giá (Uber)": round(suc_manh(dfU, c), 1),
                 "% phổ giá (Lyft)": round(suc_manh(dfL, c), 1)})
t = pd.DataFrame(rows).sort_values("% phổ giá (Lyft)", ascending=False).reset_index(drop=True)
display(t)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
d = t.sort_values("% phổ giá (Lyft)")
y = np.arange(len(d))
ax.barh(y-.2, d["% phổ giá (Uber)"], height=.38, color=MAU_HANG["Uber"], label="Uber", zorder=3)
ax.barh(y+.2, d["% phổ giá (Lyft)"], height=.38, color=MAU_HANG["Lyft"], label="Lyft", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(d["Yếu tố"], fontsize=10)
for i, (u, l) in enumerate(zip(d["% phổ giá (Uber)"], d["% phổ giá (Lyft)"])):
    ax.text(u+.7, i-.2, f"{u:.0f}%", va="center", fontsize=8)
    ax.text(l+.7, i+.2, f"{l:.0f}%", va="center", fontsize=8)
ax.set_xlabel("% phổ giá thô giải thích được")
ax.set_title("Yeu to nao quyet dinh GIA?", fontweight="bold", fontsize=13)
ax.legend(frameon=False); ax.grid(axis="y", visible=False)
fig.tight_layout(); plt.show()
print("=> LOAI XE ap dao. Cac yeu to con lai deu duoi 20%.")

**2. 🔥 Với HỆ SỐ NHÂN thì sao? (chỉ Lyft)**

In [ ]:
rows = []
for lab, c in YEU_TO:
    rows.append({"Yếu tố": lab, "cột": c,
                 "eta_với_GIÁ":   round(eta(binned(dfL[c]), dfL.price), 4),
                 "eta_với_SURGE": round(eta(binned(dfL[c]), dfL.is_surge), 4)})
ts = pd.DataFrame(rows).sort_values("eta_với_SURGE", ascending=False).reset_index(drop=True)
display(ts)

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
for a, (col, lab, c) in zip(ax, [("eta_với_GIÁ","GIA",BLUE),
                                  ("eta_với_SURGE","SURGE (Lyft)",RED)]):
    d = ts.sort_values(col)
    a.barh(range(len(d)), d[col], color=c, alpha=.88, zorder=3)
    a.set_yticks(range(len(d))); a.set_yticklabels(d["Yếu tố"], fontsize=9.5)
    for i, v in enumerate(d[col]): a.text(v+.003, i, f"{v:.3f}", va="center", fontsize=8)
    a.set_xlabel("eta (0-1)"); a.set_title(f"Suc manh voi {lab}", fontweight="bold")
    a.grid(axis="y", visible=False)
fig.tight_layout(); plt.show()

top_g = ts.sort_values("eta_với_GIÁ", ascending=False).iloc[0]
top_s = ts.iloc[0]
print(f"Manh nhat voi GIA  : {top_g['Yếu tố']} (eta={top_g['eta_với_GIÁ']:.3f})")
print(f"Manh nhat voi SURGE: {top_s['Yếu tố']} (eta={top_s['eta_với_SURGE']:.3f})")
print()
print("=> GIA va SURGE bi chi phoi boi HAI NHOM YEU TO KHAC NHAU:")
print("   GIA   <- thuoc tinh CHUYEN DI (loai xe, quang duong)")
print("   SURGE <- boi canh THI TRUONG (vi tri, thoi gian)")

**3. ⚠️ Kiểm tra độ tin cậy: yếu tố nào bị lẫn với NGÀY?**

Chỉ có 18 ngày dữ liệu. Yếu tố nào chỉ xuất hiện ở vài ngày thì kết luận không đáng tin.

In [ ]:
rows = []
for lab, c in YEU_TO:
    k = binned(df[c])
    so_ngay = df.groupby(k, observed=True)._ngay.nunique()
    rows.append({"Yếu tố": lab,
                 "số nhóm": int(k.nunique()),
                 "ngày ít nhất": int(so_ngay.min()),
                 "ngày TB": round(so_ngay.mean(), 1),
                 "đáng tin?": "⚠️ KHÔNG" if so_ngay.min() <= 5 else "✅ có"})
tc = pd.DataFrame(rows)
display(tc)
print(f"Tong so ngay co du lieu: {df._ngay.nunique()}")
print()
print("Nhom nao co 'ngay it nhat' <= 5 -> khong tach duoc khoi hieu ung ngay.")
print("Do la truong hop cua THOI TIET (Drizzle 2 ngay, Rain 3 ngay, Foggy 3 ngay).")

**4. 🔬 Hồi quy có kiểm soát — hiệu ứng thuần tính bằng %**

`log(giá) ~ log(quãng đường) + loại xe + giờ + khu vực + thời tiết`
→ hệ số cho biết ảnh hưởng riêng của từng yếu tố sau khi đã trừ các yếu tố khác.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

Cc = ["dich_vu", "gio", "khu", "thoi_tiet"]
Cn = ["log_distance", "temperature", "precipIntensity", "humidity", "windSpeed", "is_weekend"]
ket = {}
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    d = dd.sample(min(200000, len(dd)), random_state=0).copy()
    d["log_distance"] = np.log(d.distance.clip(lower=.05))
    d["gio"] = d.hour_local.astype(str); d["khu"] = d.source.astype(str)
    d["dich_vu"] = d.name.astype(str);   d["thoi_tiet"] = d.short_summary.astype(str)
    pipe = Pipeline([
        ("p", ColumnTransformer([("c", OneHotEncoder(drop="first", handle_unknown="ignore"), Cc),
                                 ("n", "passthrough", Cn)])),
        ("m", Ridge(alpha=1.0))]).fit(d[Cc+Cn], np.log(d.price))
    res = pd.DataFrame({"bien": list(pipe.named_steps["p"].get_feature_names_out()),
                        "he_so": pipe.named_steps["m"].coef_})
    res["hieu_ung_%"] = (np.exp(res.he_so) - 1) * 100
    ket[ten] = res
    print(f"{ten}: R2 hoi quy = {pipe.score(d[Cc+Cn], np.log(d.price)):.4f}")

print()
print("Bien do hieu ung THUAN (chenh giua nhom cao nhat va thap nhat):")
print(f"{'':<16}{'Uber':>10}{'Lyft':>10}")
for pre, lab in [("c__dich_vu_","Loai xe"), ("c__khu_","Khu vuc"),
                 ("c__gio_","Gio"), ("c__thoi_tiet_","Thoi tiet")]:
    v = []
    for ten in ["Uber","Lyft"]:
        s = ket[ten][ket[ten].bien.str.startswith(pre)]["hieu_ung_%"]
        v.append(f"{s.max()-s.min():.1f}%" if len(s) else "-")
    print(f"  {lab:<14}{v[0]:>10}{v[1]:>10}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
for a, ten in zip(ax, ["Uber", "Lyft"]):
    g = ket[ten][ket[ten].bien.str.startswith("c__gio_")].copy()
    g["gio"] = g.bien.str.replace("c__gio_", "", regex=False).astype(int)
    g = g.sort_values("gio")
    a.bar(g.gio, g["hieu_ung_%"],
          color=[RED if v > 0 else BLUE for v in g["hieu_ung_%"]], alpha=.85, zorder=3)
    a.axhline(0, color="#222", lw=1)
    a.set_xlabel("Gio"); a.set_ylabel("% chenh so voi 0h"); a.set_xticks(range(0, 24, 2))
    a.set_title(f"{ten} — hieu ung THUAN cua gio len gia\n"
                f"(bien do chi {g['hieu_ung_%'].max()-g['hieu_ung_%'].min():.2f} diem %)",
                fontweight="bold", fontsize=10)
    a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()
print("Sau khi kiem soat quang duong + loai xe + khu vuc + thoi tiet,")
print("gio gan nhu KHONG con anh huong gi len gia.")

**5. 📌 Kết luận & bộ feature đề xuất**

In [ ]:
print("="*72); print("TONG HOP — 4 NHOM YEU TO"); print("="*72)
print()
print("A) XEP HANG THEO % PHO GIA GIAI THICH DUOC")
for _, r in t.iterrows():
    bar = "█" * int(r["% phổ giá (Lyft)"] / 3)
    print(f"   {r['Yếu tố']:<18} {bar:<26} U={r['% phổ giá (Uber)']:>5.1f}%  L={r['% phổ giá (Lyft)']:>5.1f}%")
print()
print("B) HAI NHOM YEU TO CHO HAI TARGET KHAC NHAU")
print("   GIA   <- loai xe + quang duong   (thuoc tinh chuyen di, gan nhu tat dinh)")
print("   SURGE <- khu vuc + gio           (boi canh thi truong, bien dong)")
print()
print("C) DO TIN CAY")
kem = tc[tc["đáng tin?"].str.contains("KHÔNG")]
if len(kem):
    print("   Yeu to KHONG dang tin (bi lan voi hieu ung ngay):")
    for _, r in kem.iterrows(): print(f"     - {r['Yếu tố']} (co nhom chi xuat hien {r['ngày ít nhất']} ngay)")
else:
    print("   Tat ca yeu to deu trai deu tren nhieu ngay.")
print()
print("D) BO FEATURE DE XUAT")
print("   Model GIA   : name, distance            (+ source, destination neu can)")
print("   Model SURGE : source, destination, hour_local, short_summary")
print()
print("E) LUU Y KHI TRIEN KHAI")
print("   - Train RIENG cho tung hang (Uber va Lyft co cong thuc gia khac nhau)")
print("   - Bo cab_type (da nam trong name)")
print("   - Gio phai ma hoa PHAN LOAI/CHU KY, khong dung so nguyen 0-23")
print("   - Loai moonPhase, pressure, ozone (proxy ngay)")
print("   - Model surge chi train duoc tren Lyft (Uber toan bo = 1.0)")
print()
print("F) GIOI HAN")
print("   San nhieu ~2-3 USD khong giai thich duoc bang bat ky truong nao")
print("   -> moi model du doan gia se dung o R2 khoang 0.90-0.95")